# Motor selection  (student version)
*Written by Marc Budinger (INSA Toulouse), Scott Delbecq (ISAE-SUPAERO) and Félix Pollet (ISAE-SUPAERO), Toulouse, France.*

## Design graph

The following diagram represents the design graph of the motor’s selection. The mean speed/thrust (Ωmoy & Tmoy), the max speed/thrust (Ωmax & Tmax) and the battery voltage are assumed to be known here.

```{figure} ./assets/design_graphs/DesignGraphs_motor_student.svg
:name: design_graph_motor_student
:align: center
:width: 50%

Motor design graph
```


```{exercise}
:label: exercice_dg_propeller

- Give the main sizing problems you are able to detect.
- Propose one or multiple solutions (which can request equation manipulation, addition of design variables, addition of constraints)
- Orientate the arrows
- Give equations order, inputs/outputs at each step of this part of sizing procedure

```


### Sizing code

```{exercise}
:label: exercice_dg_motor

Propose a sizing code for the motor

```

In [17]:
import numpy as np
import scipy
import time

def sizing_code(param, arg="print"):
    k_mot = param[0]

    # Reference parameters for scaling laws
    # Motor reference
    # Ref : AXI 5325/16 GOLD LINE
    T_nom_mot_ref = 2.32  # [N.m] rated torque
    T_max_mot_ref = 85.0 / 70.0 * T_nom_mot_ref  # [N.m] max torque
    R_mot_ref = 0.03  # [Ohm] resistance
    M_mot_ref = 0.575  # [kg] mass
    K_T_ref = 0.03  # [N.m/A] torque coefficient
    T_mot_fr_ref = 0.03  # [N.m] friction torque (zero load, nominal speed)

    # Assumptions
    T_pro_hov = 0.12  # [N.m] Propeller Torque during hover
    P_pro_hov = 50  # [W] Propeller Power during hover
    Omega_pro_hov = 220.0  # [rad/s] Propeller speed during hover
    U_bat_est = 14.0  # [V] Battery voltage value (estimation)

    # Proppeler variables
    T_pro_to = 0.2  # [N.m] Propeller Torque during takeoff
    Omega_pro_to = 400 # [rad/s] Propeller speed during takeoff
    P_pro_to = 100  # [W] Propeller Power during takeoff
    T_pro_vol = 0.15
    Omega_pro_vol = 300


    # Design variables
    T_max_mot = max(T_pro_to, T_pro_vol)
    Omega_max_mot = max(Omega_pro_to, Omega_pro_vol)
    P_max = max(T_pro_to*Omega_pro_to, T_pro_vol*Omega_pro_vol)
    T_nom_mot = T_nom_mot_ref * (T_max_mot / T_max_mot_ref)
    K_T = (1/k_mot) * (U_bat_est / Omega_max_mot)
    I = T_max_mot / K_T
    R_mot = R_mot_ref * ((K_T / K_T_ref)**(2)) * ((T_nom_mot / T_nom_mot_ref)**(-5/3.5))
    M_mot = M_mot_ref * (T_nom_mot / T_nom_mot_ref)**(3/3.5)
    U_bat = K_T * Omega_max_mot + R_mot * I

    # Objective = motor mass
    objective = k_mot

    # Constraints (should be >=0)
    C1 = U_bat_est - K_T * Omega_max_mot - R_mot * I

    # Objective and constraints
    if arg == "objective":
        return objective
    if arg == "objectiveP":
        if (C1 < 0.0):
            # If constraints are not respected we penalize
            # the objective for contraint less algorithms
            return objective * 1e5
        else:
            return objective
    elif arg == "print":
        print("Objective:")
        print("     Total mass = %.2f kg" % M_mot)
        print("     C1 = %.2f V" % C1)
        print("     U_bat = %.2f V" % U_bat)
        print("     I = %.2f A" % I)
        print("     k_mot = %.2f A" % k_mot)
    elif arg == "constraints":
        return C1
    else:
        raise TypeError(
            "Unknown argument for sizing_code: use 'print', 'objective', 'objectiveP' or 'contraints'"
        )


# Initial values vector for design variables
parameters = np.array((0.5,0))

# Initial characteristics before optimization
print("-----------------------------------------------")
print("Initial characteristics before optimization :")

sizing_code(parameters, "print")
print("-----------------------------------------------")

# Resolution of the problem

contrainte = lambda x: sizing_code(x, "constraints")
objectif = lambda x: sizing_code(x, "objective")

start = time.time()
result = scipy.optimize.fmin_slsqp(
    func=objectif,
    x0=parameters,
    bounds=[
        (0, 2),
        (0 ,0)
    ],
    f_ieqcons=contrainte,
    iter=1000,
    acc=1e-5,
)
end = time.time()

# Final characteristics after optimization
print("-----------------------------------------------")
print("Final characteristics after optimization :")

sizing_code(result, "print")
print("-----------------------------------------------")
print("Calculation time:\n", end - start, "s")

-----------------------------------------------
Initial characteristics before optimization :
Objective:
     Total mass = 0.06 kg
     C1 = -34.42 V
     U_bat = 48.42 V
     I = 2.86 A
     k_mot = 0.50 A
-----------------------------------------------
Inequality constraints incompatible    (Exit mode 4)
            Current function value: 0.5
            Iterations: 1
            Function evaluations: 3
            Gradient evaluations: 1
-----------------------------------------------
Final characteristics after optimization :
Objective:
     Total mass = 0.06 kg
     C1 = -34.42 V
     U_bat = 48.42 V
     I = 2.86 A
     k_mot = 0.50 A
-----------------------------------------------
Calculation time:
 0.001773834228515625 s


/usr/local/lib/python3.12/dist-packages/scipy/optimize/_numdiff.py:687: RuntimeWarning: invalid value encountered in divide
  df_dx = [delf / delx for delf, delx in zip(df, dx)]


In [ ]:
# Equations

## To be completed

In [ ]:
%whos